# Train YOLO on Naruto Hand Sign Dataset
This notebook trains an Ultralytics YOLO object detection model using the dataset in `naruto-hand-sign-4`.
It also validates the model and runs a quick prediction preview.

In [ ]:
# Install required packages (run once per environment)
# %pip install -q ultralytics pyyaml

: 

In [3]:
from pathlib import Path
import os
import warnings
import torch
import yaml
from ultralytics import YOLO

PROJECT_DIR = Path.cwd()
DATASET_DIR = PROJECT_DIR / "naruto-hand-sign-4"
DATA_YAML = DATASET_DIR / "data.yaml"

assert DATASET_DIR.exists(), f"Dataset folder not found: {DATASET_DIR}"
assert DATA_YAML.exists(), f"data.yaml not found: {DATA_YAML}"

MPS_AVAILABLE = torch.backends.mps.is_available() and torch.backends.mps.is_built()
if MPS_AVAILABLE:
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = 0
else:
    DEVICE = "cpu"

CPU_CORES = os.cpu_count() or 8
# Keep a couple of CPU cores free so UI/system stay responsive while training.
WORKERS = max(2, min(12, CPU_CORES - 2))

# Speed-oriented defaults for Apple Silicon training.
MODEL_WEIGHTS = "yolov8s.pt"
TRAIN_EPOCHS = 100
TRAIN_IMGSZ = 640
TRAIN_BATCH = 32 if DEVICE == "mps" else 16

with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    torch.set_num_threads(CPU_CORES)
torch.set_float32_matmul_precision("high")

print(f"Project dir : {PROJECT_DIR}")
print(f"Dataset dir : {DATASET_DIR}")
print(f"Data config : {DATA_YAML}")
print(f"Train device: {DEVICE}")
print(f"MPS built   : {torch.backends.mps.is_built()}")
print(f"MPS avail   : {torch.backends.mps.is_available()}")
print(f"CPU cores   : {CPU_CORES}")
print(f"Workers     : {WORKERS}")
print(f"Model       : {MODEL_WEIGHTS}")
print(f"Batch size  : {TRAIN_BATCH}")

if DEVICE != "mps":
    print("WARNING: MPS is not active. Training will not fully use your Apple GPU.")

Project dir : /Users/farrellhrs/Documents/CBL/CBL_1/hand_gesture_model
Dataset dir : /Users/farrellhrs/Documents/CBL/CBL_1/hand_gesture_model/naruto-hand-sign-4
Data config : /Users/farrellhrs/Documents/CBL/CBL_1/hand_gesture_model/naruto-hand-sign-4/data.yaml
Train device: mps
MPS built   : True
MPS avail   : True
CPU cores   : 10
Workers     : 8
Model       : yolov8s.pt
Batch size  : 32


In [ ]:
# Build a local YAML with absolute image paths (robust to train/val/valid folder naming)
with open(DATA_YAML, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

def resolve_image_dir(split_key: str, yaml_value: str):
    raw = Path(str(yaml_value))
    candidates = []

    if raw.is_absolute():
        candidates.append(raw)

    candidates.extend(
        [
            (DATA_YAML.parent / raw),
            (DATASET_DIR / raw),
        ]
    )

    split_aliases = {
        "train": ["train"],
        "val": ["valid", "val"],
        "test": ["test"],
    }[split_key]

    for name in split_aliases:
        candidates.append(DATASET_DIR / name / "images")
        candidates.append(DATASET_DIR / name)

    tried = []
    seen = set()
    for p in candidates:
        rp = p.resolve()
        key = str(rp)
        if key in seen:
            continue
        seen.add(key)
        tried.append(rp)

        if rp.exists() and rp.is_dir():
            images_subdir = rp / "images"
            if images_subdir.exists() and images_subdir.is_dir():
                return images_subdir.resolve(), tried
            return rp, tried

    return None, tried

resolved_paths = {}
for split in ["train", "val", "test"]:
    resolved, tried = resolve_image_dir(split, cfg.get(split, ""))
    if resolved is None:
        tried_text = "\n".join(f" - {p}" for p in tried)
        raise FileNotFoundError(
            f"Could not resolve image directory for split '{split}' with value '{cfg.get(split)}'.\n"
            f"Tried:\n{tried_text}"
        )
    resolved_paths[split] = resolved
    print(f"{split}: {resolved} (exists={resolved.exists()})")

LOCAL_DATA_YAML = PROJECT_DIR / "naruto_hand_sign_data_local.yaml"
local_cfg = dict(cfg)
local_cfg["train"] = str(resolved_paths["train"])
local_cfg["val"] = str(resolved_paths["val"])
local_cfg["test"] = str(resolved_paths["test"])

with open(LOCAL_DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(local_cfg, f, sort_keys=False, allow_unicode=False)

print(f"\nSaved local training config: {LOCAL_DATA_YAML}")
print(f"Classes ({local_cfg['nc']}): {local_cfg['names']}")

train: /Users/farrellhrs/Documents/CBL/CBL_1/hand_gesture_model/naruto-hand-sign-4/train/images (exists=True)
val: /Users/farrellhrs/Documents/CBL/CBL_1/hand_gesture_model/naruto-hand-sign-4/valid/images (exists=True)
test: /Users/farrellhrs/Documents/CBL/CBL_1/hand_gesture_model/naruto-hand-sign-4/test/images (exists=True)

Saved local training config: /Users/farrellhrs/Documents/CBL/CBL_1/hand_gesture_model/naruto_hand_sign_data_local.yaml
Classes (12): ['bird', 'boar', 'dog', 'dragon', 'hare', 'horse', 'monkey', 'ox', 'ram', 'rat', 'snake', 'tiger']


In [ ]:
# Optional quick dataset size check
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
for split in ["train", "val", "test"]:
    img_dir = resolved_paths[split]
    n_images = sum(1 for p in img_dir.iterdir() if p.suffix.lower() in image_exts) if img_dir.exists() else 0
    print(f"{split:>5}: {n_images} images")

train: 4299 images
  val: 1228 images
 test: 615 images


In [ ]:
# Train YOLO (Apple Silicon tuned)
if DEVICE != "mps":
    raise RuntimeError(
        "MPS is not active. Please select a native Apple Silicon Python env so training uses your M5 GPU."
    )

model = YOLO(MODEL_WEIGHTS)

train_results = model.train(
    data=str(LOCAL_DATA_YAML),
    epochs=TRAIN_EPOCHS,
    imgsz=TRAIN_IMGSZ,
    batch=TRAIN_BATCH,
    device=DEVICE,
    project=str(DATASET_DIR / "phase1_outputs_unified"),
    name="train3_yolov8s_m5",
    exist_ok=True,
    patience=20,
    workers=WORKERS,
    pretrained=True,
    verbose=True,
    plots=True,
    seed=42,
    deterministic=False,
    save=True,
    val=True,
    cache="ram",
    amp=True,
    lr0=0.01,
    optimizer="auto",
)

print(f"Training outputs saved to: {train_results.save_dir}")

Ultralytics 8.4.37 🚀 Python-3.11.15 torch-2.2.2 MPS (Apple M5)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/farrellhrs/Documents/CBL/CBL_1/hand_gesture_model/naruto_hand_sign_data_local.yaml, degrees=0.0, deterministic=False, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train3_yolov8s_m5, nbs=64, nms=False, opset=None, optimize=False

/Users/farrellhrs/miniconda3/envs/cv_env/lib/python3.11/site-packages/ultralytics/utils/torch_utils.py:255: UserWarning: Cannot set number of intraop threads after parallel work has started or after set_num_threads call when using native parallel backend (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1711403251597/work/aten/src/ATen/ParallelNative.cpp:228.)
  torch.set_num_threads(NUM_THREADS)  # reset OMP_NUM_THREADS for cpu training


Overriding model.yaml nc=80 with nc=12

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     29056  ultralytics.nn.modules.block.C2f             [64, 64, 1, True]             
  3                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  4                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  5                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  6                  -1  2    788480  ultralytics.nn.modules.block.C2f             [256, 256, 2, True]           
  7                  -1  1   1180672  ultralytic

KeyboardInterrupt: 

In [ ]:
# Validate best checkpoint
best_weights = Path(train_results.save_dir) / "weights" / "best.pt"
assert best_weights.exists(), f"Best checkpoint not found: {best_weights}"

best_model = YOLO(str(best_weights))
metrics = best_model.val(data=str(LOCAL_DATA_YAML), split="val", device=DEVICE)

print(f"Best checkpoint: {best_weights}")
print(metrics)

In [ ]:
# Run quick predictions on a few test images and display saved outputs
from IPython.display import Image, display

test_dir = resolved_paths["test"]
test_images = [
    p for p in sorted(test_dir.iterdir())
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
][:5]

assert test_images, f"No test images found in: {test_dir}"

pred_project = DATASET_DIR / "phase1_outputs_unified"
pred_name = "train3_preview"

best_model.predict(
    source=[str(p) for p in test_images],
    conf=0.25,
    save=True,
    project=str(pred_project),
    name=pred_name,
    exist_ok=True,
    device=DEVICE,
    verbose=False,
)

pred_dir = pred_project / pred_name
print(f"Prediction images saved to: {pred_dir}")

pred_images = [
    p for p in sorted(pred_dir.iterdir())
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
][:5]

for p in pred_images:
    display(Image(filename=str(p)))

NameError: name 'resolved_paths' is not defined

## Run Order
1. Run all cells from top to bottom.
2. Training artifacts are saved under `naruto-hand-sign-4/phase1_outputs_unified/train3_yolov8n`.
3. Preview predictions are saved under `naruto-hand-sign-4/phase1_outputs_unified/train3_preview`.